In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os
from pathlib import Path

import numpy as np
from models_under_pressure.interfaces.dataset import LabelledDataset

/Users/ep/Documents/research/vrcp_proj/reliable-llm-monitoring/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
LAYER = 11
DATA_DIR = os.environ["DATA_DIR"]

In [5]:
from dataset import ActivationConfig

activation_config = ActivationConfig(model_name=MODEL_NAME, layer=LAYER)

# Cascade

In [6]:
from dataset import load_dataset, sample_from_dataset

train_dataset = sample_from_dataset(
    load_dataset(
        Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
        activation_config=activation_config,
    ),
    1000,
)

# calibration and test are both exchangeable (actually i.i.d.)
calibration_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/dev/toolace_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/dev/anthropic_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
test_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/test/toolace_test_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/test/anthropic_test_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)

In [7]:
from dataset import split_dataset

opt_dataset, calibration_dataset = split_dataset(calibration_dataset, [0.5, 0.5])

In [8]:
from probes import train_probe

probe = train_probe(train_dataset)

100%|██████████| 4/4 [00:00<00:00,  9.71it/s]


In [9]:
from cascade import CascadePredictionResults
from sklearn.metrics import accuracy_score, roc_auc_score


def baseline_budget_cost(cascade_scores: CascadePredictionResults) -> float:
    """Rate at which you call the baseline"""
    return cascade_scores.used_baseline.mean()


def empirical_roc_auc(cascade_scores: CascadePredictionResults, dataset: LabelledDataset) -> float:
    """Empirical performance of the cascade."""
    return roc_auc_score(
        dataset.labels_numpy(),
        cascade_scores.final_scores,
    )


def empirical_accuracy(cascade_scores: CascadePredictionResults, dataset: LabelledDataset) -> float:
    """Empirical accuracy of the cascade."""
    predicted_labels = (cascade_scores.final_scores >= 0.5).astype(int)
    return accuracy_score(
        dataset.labels_numpy(),
        predicted_labels,
    )

In [10]:
from cascade import run_cascade
from probes import probe_function

thresholds = np.linspace(0, 1, 11)[1:-1]  # Exclude 0 and 1
budget_scores = []
empirical_performance_scores = []

for i, t in enumerate(thresholds):
    print(f"Evaluating threshold: {t:.2f}, {i} of {len(thresholds)}")
    opt_scores = run_cascade(
        probe=lambda dataset: probe_function(probe, dataset),
        baseline_model_name=MODEL_NAME,
        threshold=t,
        dataset=opt_dataset,
        merge_strategy="avg",
    )
    budget_scores.append(baseline_budget_cost(opt_scores))
    empirical_performance_scores.append(empirical_accuracy(opt_scores, opt_dataset))

Evaluating threshold: 0.10, 0 of 9


100%|██████████| 2/2 [00:01<00:00,  1.93it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!
0it [00:00, ?it/s]


Evaluating threshold: 0.20, 1 of 9


100%|██████████| 2/2 [00:00<00:00,  4.37it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
0it [00:00, ?it/s]


Evaluating threshold: 0.30, 2 of 9


...

100%|██████████| 2/2 [00:03<00:00,  1.60s/it]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
100%|██████████| 8/8 [01:01<00:00,  7.63s/it]


Evaluating threshold: 0.80, 7 of 9


100%|██████████| 2/2 [00:04<00:00,  2.15s/it]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
100%|██████████| 13/13 [02:15<00:00, 10.39s/it]


Evaluating threshold: 0.90, 8 of 9


100%|██████████| 2/2 [00:04<00:00,  2.43s/it]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
100%|██████████| 20/20 [03:29<00:00, 10.46s/it]


In [ ]:
from ltt_utils import is_pareto

costs = np.concatenate(
    [
        -np.array(budget_scores).reshape(-1, 1),  # we want to minimise the budget cost
        np.array(empirical_performance_scores).reshape(-1, 1),
    ],
    axis=1,
)


pareto_front = is_pareto(-costs)
pareto_front

array([ True,  True,  True,  True,  True,  True,  True,  True,  True])

In [12]:
costs

array([[-0.        ,  0.6536965 ],
       [-0.        ,  0.6536965 ],
       [-0.        ,  0.6536965 ],
       [-0.        ,  0.6536965 ],
       [-0.        ,  0.6536965 ],
       [-0.06031128,  0.67120623],
       [-0.11867704,  0.6770428 ],
       [-0.19844358,  0.68093385],
       [-0.29961089,  0.68093385]])

In [ ]:
import plotly.express as px

fig = px.scatter(
    x=-costs[:, 0],  # Budget cost (negative because we minimized -cost)
    y=costs[:, 1],  # Empirical performance
    title="Budget Cost vs Empirical Performance",
    labels={"x": "Budget Cost", "y": "Empirical Performance"},
)
fig.add_scatter(
    x=-costs[pareto_front, 0],
    y=costs[pareto_front, 1],
    mode="markers",
    marker=dict(color="red", size=10, symbol="star"),
    name="Pareto Front",
)
fig.show()

In [15]:
budget_scores

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.12804878048780488,
 0.27439024390243905,
 0.4573170731707317,
 0.6524390243902439]

In [21]:
empirical_performance_scores

[0.9779911964785915,
 0.9779911964785915,
 0.9779911964785915,
 0.9779911964785915,
 0.9779911964785915,
 0.9771908763505402,
 0.9771908763505402,
 0.9735894357743098,
 0.9679871948779513]

In [30]:
# why does the baseline suck?

In [ ]:
# now define the empirical risk functions that we will use to find the pareto front and run LTT

# LTT

In [ ]:
# given train, validation, calibration and test splits
# train probes on `train`
# e2e score pipeline: give validation dataset and threshold, get final performance metrics (cost + score)
# vanilla threshold estimation using calibration set once
# LTT using above e2e pipeline and calibration set + pareto frontier etc...
# comapre e2e performance distributions across multiple runs

In [ ]:
# make e2e pipeline with final performance score + cost as a function of dataset splits
# find pareto front
# run fixed sequence testing